# 🗄️ Nucleotide Sequence Databases

---

## Learning Objectives

- Navigate NCBI GenBank and other sequence databases
- Understand sequence file formats (FASTA, GenBank)
- Retrieve sequences programmatically
- Parse and analyze genomic annotations

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This notebook is designed to bridge concept to practical analysis workflows.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Inspect assumptions before running model/statistical code.
- Track input/output files for reproducibility.
- Interpret outputs with biological context, not numbers alone.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


---

## Major Sequence Databases

```
┌─────────────────────────────────────────────────────────────────────────┐
│                    INTERNATIONAL NUCLEOTIDE DATABASES                   │
├─────────────────────────────────────────────────────────────────────────┤
│                                                                         │
│   ┌──────────────┐    ┌──────────────┐    ┌──────────────┐             │
│   │    NCBI      │    │     ENA      │    │    DDBJ      │             │
│   │   GenBank    │◄──►│   (EMBL)     │◄──►│   (Japan)    │             │
│   │    (USA)     │    │  (Europe)    │    │              │             │
│   └──────────────┘    └──────────────┘    └──────────────┘             │
│          │                   │                   │                      │
│          └───────────────────┼───────────────────┘                      │
│                              │                                          │
│                    Daily synchronization                                │
│                 (INSDC collaboration)                                   │
│                                                                         │
│   All three databases share the same data!                              │
│                                                                         │
└─────────────────────────────────────────────────────────────────────────┘
```

---

## GenBank File Format

```
LOCUS       AB000263     368 bp    mRNA    linear   PRI 05-FEB-1999
DEFINITION  Homo sapiens mRNA for prepro cortistatin like peptide...
ACCESSION   AB000263
VERSION     AB000263.1  GI:1754602
KEYWORDS    .
SOURCE      Homo sapiens (human)
  ORGANISM  Homo sapiens
            Eukaryota; Metazoa; Chordata; Craniata; ...
REFERENCE   1  (bases 1 to 368)
  AUTHORS   Fukusumi,S., Kitada,C., Takekawa,S., ...
  TITLE     Identification and characterization of a novel...
  JOURNAL   Biochem. Biophys. Res. Commun. 232 (1), 157-163
FEATURES             Location/Qualifiers
     source          1..368
                     /organism="Homo sapiens"
                     /mol_type="mRNA"
     CDS             43..180
                     /product="prepro-cortistatin like peptide"
                     /protein_id="BAA19052.1"
ORIGIN
        1 atggcctctg cagcagacga ccgatgcaag cagccggtcc tgcagccgct
       61 gcagcagccg gagcagctgc gcgcgctgga gccgggccag ctgaccgagg
//
```

In [ ]:
from Bio import Entrez, SeqIO

# Always set your email for NCBI queries
Entrez.email = "your.email@example.com"

def fetch_genbank_record(accession):
    """
    Fetch GenBank record from NCBI.
    
    Parameters:
        accession: GenBank accession number (e.g., 'NM_001301717')
        
    Returns:
        SeqRecord object
    """
    handle = Entrez.efetch(
        db="nucleotide",
        id=accession,
        rettype="gb",
        retmode="text"
    )
    record = SeqIO.read(handle, "genbank")
    handle.close()
    return record

def search_ncbi(query, database="nucleotide", max_results=10):
    """
    Search NCBI database.
    
    Parameters:
        query: Search terms
        database: NCBI database name
        max_results: Maximum number of results
        
    Returns:
        List of accession IDs
    """
    handle = Entrez.esearch(
        db=database,
        term=query,
        retmax=max_results
    )
    record = Entrez.read(handle)
    handle.close()
    return record['IdList']

print("NCBI access functions defined.")
print("\nExample usage:")
print('  ids = search_ncbi("human insulin[Gene] AND mRNA[Filter]")')
print('  record = fetch_genbank_record("NM_000207")')

---

## FASTA Format

```
>gi|1234567|gb|AB000263.1| Homo sapiens mRNA for prepro cortistatin
ATGGCCTCTGCAGCAGACGACCGATGCAAGCAGCCGGTCCTGCAGCCGCTGCAGCAGCCG
GAGCAGCTGCGCGCGCTGGAGCCGGGCCAGCTGACCGAGGCCTGCTGGCCCTGA

Header line structure:
>database|identifier|accession.version| description

Rules:
• Header starts with >
• Sequence follows on next line(s)
• No spaces in sequence
• Line length typically 60-80 characters
• Multiple sequences separated by > header
```

In [ ]:
def parse_fasta(fasta_text):
    """
    Parse FASTA format text.
    
    Parameters:
        fasta_text: String containing FASTA formatted sequences
        
    Returns:
        List of (header, sequence) tuples
    """
    sequences = []
    current_header = None
    current_seq = []
    
    for line in fasta_text.strip().split('\n'):
        line = line.strip()
        if line.startswith('>'):
            if current_header is not None:
                sequences.append((current_header, ''.join(current_seq)))
            current_header = line[1:]  # Remove >
            current_seq = []
        else:
            current_seq.append(line)
    
    if current_header is not None:
        sequences.append((current_header, ''.join(current_seq)))
    
    return sequences

def write_fasta(sequences, output_file, line_width=60):
    """
    Write sequences to FASTA file.
    
    Parameters:
        sequences: List of (header, sequence) tuples
        output_file: Output filename
        line_width: Characters per line for sequence
    """
    with open(output_file, 'w') as f:
        for header, seq in sequences:
            f.write(f">{header}\n")
            for i in range(0, len(seq), line_width):
                f.write(seq[i:i+line_width] + '\n')

# Example
example_fasta = """>seq1 Human insulin
MALWMRLLPLLALLALWGPDPAAA
>seq2 Mouse insulin
MALWTRLLPLLALLALWAPAPAHA
"""

parsed = parse_fasta(example_fasta)
for header, seq in parsed:
    print(f"Header: {header}")
    print(f"Sequence: {seq}")
    print()

---

## Database Identifiers

| Database | Format | Example |
|----------|--------|--------|
| GenBank | XX######.# | AB000263.1 |
| RefSeq (mRNA) | NM_###### | NM_000207.2 |
| RefSeq (protein) | NP_###### | NP_000198.1 |
| RefSeq (genome) | NC_###### | NC_000001.11 |
| UniProt | P##### or Q##### | P01308 |
| PDB | #### | 1A2B |

---

## 🏋️ Exercises

### Exercise 1: Retrieve Sequences
Search for and download all human hemoglobin mRNA sequences.

### Exercise 2: Extract Features
Parse GenBank files to extract CDS coordinates and translations.

### Exercise 3: Build Local Database
Create a local FASTA file from multiple GenBank records.

---

## 📚 Resources

- [NCBI GenBank](https://www.ncbi.nlm.nih.gov/genbank/)
- [ENA](https://www.ebi.ac.uk/ena/browser/home)
- [BioPython Entrez](https://biopython.org/wiki/Entrez)